# ?? SirenCast UA: Live Airspace Threat Forecasting Demo
**Multi-modal 30-minute early warning air raid risk forecasting for Kyiv.**

This interactive notebook demonstrates the **SirenCast UA Live Forecasting Engine**:
1. **Live Data Ingestion**: Open Ubilling Air Alert Map API (with rate-limiting cache) + zero-token Telegram channel threat scraper.
2. **Real-Time Risk Inference**: Evaluates current live telemetry against the trained multimodal LightGBM model ($t \to t+30\text{min}$).
3. **What-If Scenario Sandbox**: Simulates escalation events (Shahed launches, border alerts, Kyiv active siren) to visualize how risk probabilities shift.


In [ ]:
import sys
import os
import pandas as pd
sys.path.insert(0, os.path.abspath("."))

from src.live_data import parse_live_siren_states, fetch_live_telegram_stream
from src.live_forecast import predict_live_risk, format_cli_dashboard

print("Environment initialized. Ready for live inference.")


## 1. Live Data Ingestion
Fetch current live siren states across all Ukrainian regions and scrape threat keywords from public defense channels (`@war_monitor`, `@kpszsu`).


In [ ]:
alerts = parse_live_siren_states()
print(f"Timestamp: {alerts['timestamp_utc'].strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"National Active Sirens: {alerts['national_active_count']} / 25 regions")
print(f"Kyiv City Active: {'?? ACTIVE' if alerts['regions']['Kyiv City'] else '?? Clear'}")

print("\nKyivska Oblast Surrounding Raions:")
for raion, active in alerts['kyiv_raions'].items():
    status = "?? ACTIVE" if active else "?? Clear"
    print(f"  - {raion:<20}: {status}")

tg_msgs = fetch_live_telegram_stream()
print(f"\nTelegram Stream: {len(tg_msgs)} recent messages scraped without tokens.")
for m in tg_msgs[:3]:
    safe_text = m['text'][:85].encode('ascii', errors='backslashreplace').decode('ascii')
    print(f"  [@{m['channel']}]: {safe_text}...")


## 2. Live Multimodal Inference
Run the trained LightGBM model on real-time data to compute Kyiv's air raid risk for the next 30 minutes ($[t, t+30\text{min}]$).


In [ ]:
result = predict_live_risk()

print(format_cli_dashboard(result))


## 3. What-If Scenario Sandbox
Simulate how the model responds to hypothetical threat escalation scenarios in real time.


In [ ]:
from datetime import datetime, timezone

scenarios = [
    {
        "name": "1. Calm Skies (Quiescent baseline)",
        "alerts": {
            "regions": {"Kyiv City": False, "Chernihivska oblast": False, "Sumska oblast": False},
            "kyiv_raions": {r: False for r in alerts['kyiv_raions']},
            "timestamp_utc": datetime.now(timezone.utc),
            "national_active_count": 0
        },
        "texts": ["?????? ?????????? ???????. ???? ?????."]
    },
    {
        "name": "2. Shahed Group Approaching Border (Chernihiv/Sumy)",
        "alerts": {
            "regions": {"Kyiv City": False, "Chernihivska oblast": True, "Sumska oblast": True},
            "kyiv_raions": {r: False for r in alerts['kyiv_raions']},
            "timestamp_utc": datetime.now(timezone.utc),
            "national_active_count": 4
        },
        "texts": [
            "????? ??????? ???? Shahed ?????? ?? ????????????!",
            "?????? ????? ??????? ? ???????? ???????"
        ]
    },
    {
        "name": "3. Immediate Vector to Kyiv / Brovary",
        "alerts": {
            "regions": {"Kyiv City": False, "Chernihivska oblast": True, "Kyivska oblast": True},
            "kyiv_raions": {"Brovarskyi raion": True, "Boryspilskyi raion": True},
            "timestamp_utc": datetime.now(timezone.utc),
            "national_active_count": 6
        },
        "texts": [
            "?????! ???? Shahed ???? ??????? ?????? ?? ????!",
            "???????? ? ?????? ??? ?? ???????"
        ]
    },
    {
        "name": "4. Ballistic / Hypersonic Threat Warning",
        "alerts": {
            "regions": {"Kyiv City": False, "Chernihivska oblast": True, "Sumska oblast": True, "Kyivska oblast": True},
            "kyiv_raions": {r: True for r in alerts['kyiv_raions']},
            "timestamp_utc": datetime.now(timezone.utc),
            "national_active_count": 12
        },
        "texts": [
            "????????? ???? ?? ???? ????? ????????????!",
            "??????? ???????????? ???????????? ????????? ? ???????!"
        ]
    },
    {
        "name": "5. Kyiv City Siren Already Active (Continuation)",
        "alerts": {
            "regions": {"Kyiv City": True, "Kyivska oblast": True, "Chernihivska oblast": True},
            "kyiv_raions": {r: True for r in alerts['kyiv_raions']},
            "timestamp_utc": datetime.now(timezone.utc),
            "national_active_count": 14
        },
        "texts": [
            "????????? ??????? ? ????? ??????. ?????????? ??????????? ????????."
        ]
    }
]

records = []
for sc in scenarios:
    res = predict_live_risk(simulated_alerts=sc["alerts"], simulated_telegram_texts=sc["texts"])
    records.append({
        "Scenario": sc["name"],
        "Threat Probability": f"{res['probability_pct']:.1f}%",
        "Risk Level": res["risk_level"],
        "Top Driver": res["top_drivers"][0] if res["top_drivers"] else "None"
    })

pd.DataFrame(records)
